In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix


In [165]:
dt =  pd.read_csv('datasets/subset.csv')

In [ ]:
#Remover as colunas inúteis da base subset(Original)
#BaseOLimpa = a base original, porém retirando essas colunas
BaseOLimpa = dt.drop(["timestamp_created", "language", "review", "comment_count", "author.num_reviews", "author.num_games_owned"], axis= 1)


<bound method DataFrame.info of          app_id                  app_name  recommended  votes_helpful  \
0        292030  The Witcher 3: Wild Hunt         True              0   
1        292030  The Witcher 3: Wild Hunt         True              0   
2        292030  The Witcher 3: Wild Hunt         True              0   
3        292030  The Witcher 3: Wild Hunt         True              0   
4        292030  The Witcher 3: Wild Hunt         True              0   
...         ...                       ...          ...            ...   
5901416  546560           Half-Life: Alyx         True              1   
5901417  546560           Half-Life: Alyx         True              1   
5901418  546560           Half-Life: Alyx         True              2   
5901419  546560           Half-Life: Alyx         True              0   
5901420  546560           Half-Life: Alyx         True              0   

            author.steamid  author.playtime_at_review  
0        76561198170193529         

In [ ]:
#Base Games é a base de onde consegi as categorias https://www.kaggle.com/datasets/fronkongames/steam-games-dataset/data
BaseGames = pd.read_csv("datasets/games.csv")

In [104]:
#Base Games após limpeza de colunas
Games = BaseGames[['AppID', 'Categories', "Genres", "Tags"]]

In [106]:
#Nulos da base Games
Games.isnull().sum()

AppID             6
Categories     5913
Genres         4841
Tags          29763
dtype: int64

In [145]:
#Remoção de nulos da coluna AppID
Games = Games.dropna(subset=["AppID"])
Games.isnull().sum()

AppID             0
Categories     5910
Genres         4839
Tags          29757
dtype: int64

In [152]:
#Remoção das linhas onde as 3 colunas estão com nulo(ao mesmo tempo)
GamesLimpeza = Games.dropna(subset = ["Categories", "Genres", "Tags"], how = 'all')

In [153]:
GamesLimpeza.isnull().sum()

AppID             0
Categories     1242
Genres          171
Tags          25089
dtype: int64

In [154]:
#Concatenar as 2 bases
gamesteste = GamesLimpeza.rename(columns={'AppID': 'app_name'})


In [155]:
# Fazer o merge das bases
BaseFinal = pd.merge(BaseOLimpa, gamesteste[['app_name', 'Categories', 'Genres', 'Tags']], on='app_name', how='left')


In [156]:
BaseFinal

,app_id,app_name,recommended,votes_helpful,author.steamid,author.playtime_at_review,Categories,Genres,Tags
0,292030,The Witcher 3: Wild Hunt,True,0,76561198170193529,823.0,NaN,NaN,NaN
1,292030,The Witcher 3: Wild Hunt,True,0,76561198119302812,4192.0,NaN,NaN,NaN
2,292030,The Witcher 3: Wild Hunt,True,0,76561198284845223,2518.0,NaN,NaN,NaN
3,292030,The Witcher 3: Wild Hunt,True,0,76561198370568524,517.0,NaN,NaN,NaN
4,292030,The Witcher 3: Wild Hunt,True,0,76561198040150323,165.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
6010648,546560,Half-Life: Alyx,True,1,76561198824257821,2230.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure","VR,FPS,Story Rich,Horror,Female Protagonist,Sh..."
6010649,546560,Half-Life: Alyx,True,1,76561198003907172,1083.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure","VR,FPS,Story Rich,Horror,Female Protagonist,Sh..."
6010650,546560,Half-Life: Alyx,True,2,76561198020563704,1513.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure","VR,FPS,Story Rich,Horror,Female Protagonist,Sh..."
6010651,546560,Half-Life: Alyx,True,0,76561198032433284,752.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure","VR,FPS,Story Rich,Horror,Female Protagonist,Sh..."


In [ ]:
#Retirar as 3 colunas da BaseFinal
teste = BaseFinal.dropna(subset = ["Categories", "Genres", "Tags"], how = 'all')

NameError: name 'BaseFinal' is not defined

In [ ]:
#Retirar a coluna Tags da base teste
Base = teste.drop(["Tags"], axis=1)

In [162]:
Base

,app_id,app_name,recommended,votes_helpful,author.steamid,author.playtime_at_review,Categories,Genres
72593,70,Half-Life,False,0,76561198352346016,530.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
72594,70,Half-Life,True,0,76561198074088375,237.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
72595,70,Half-Life,True,0,76561198945746887,3297.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
72596,70,Half-Life,True,0,76561198911822629,1015.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
72597,70,Half-Life,True,0,76561198956790263,1145.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
...,...,...,...,...,...,...,...,...
6010648,546560,Half-Life: Alyx,True,1,76561198824257821,2230.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
6010649,546560,Half-Life: Alyx,True,1,76561198003907172,1083.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
6010650,546560,Half-Life: Alyx,True,2,76561198020563704,1513.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
6010651,546560,Half-Life: Alyx,True,0,76561198032433284,752.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"


In [163]:
Base.to_csv('datasets/Base.csv', index=False)

In [11]:
Base = pd.read_csv('datasets/Base.csv')

In [12]:
Base

,app_id,app_name,recommended,votes_helpful,author.steamid,author.playtime_at_review,Categories,Genres
0,70,Half-Life,False,0,76561198352346016,530.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
1,70,Half-Life,True,0,76561198074088375,237.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
2,70,Half-Life,True,0,76561198945746887,3297.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
3,70,Half-Life,True,0,76561198911822629,1015.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
4,70,Half-Life,True,0,76561198956790263,1145.0,"Single-player,Multi-player,PvP,Online PvP,Stea...",Action
...,...,...,...,...,...,...,...,...
4963320,546560,Half-Life: Alyx,True,1,76561198824257821,2230.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
4963321,546560,Half-Life: Alyx,True,1,76561198003907172,1083.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
4963322,546560,Half-Life: Alyx,True,2,76561198020563704,1513.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"
4963323,546560,Half-Life: Alyx,True,0,76561198032433284,752.0,"Single-player,Steam Achievements,Captions avai...","Action,Adventure"


In [13]:
#Converto a coluna Categories em colunas binárias para cada categoria
categories_dummies = Base['Categories'].str.get_dummies(sep=',')
Base2 = pd.concat([Base, categories_dummies], axis=1)


In [14]:
#Retira a coluna Categories da Base2
Base2 = Base2.drop(['Categories'], axis=1)

In [15]:
#Trasforma a coluna Genres em variáveis dummy
Base3 = Base2['Genres'].str.get_dummies(sep=',')
Base3 = pd.concat([Base2, Base3], axis=1)


In [16]:
Base3 = Base3.drop(['Genres'], axis=1)

In [6]:
Base3 = pd.read_csv('datasets/Base3.csv')

In [33]:
# Conta o número de interações por usuário
user_review_counts = Base3['author.steamid'].value_counts()

# Seleciona apenas os usuários com mais de 3 interações
users_to_keep = user_review_counts[user_review_counts > 3].index

# Filtra o train_data para manter apenas os usuários com mais de 3 interações
Base4 = Base3[Base3['author.steamid'].isin(users_to_keep)].reset_index(drop=True)

# Verifica o resultado
print(f"Tamanho original do train_data: {Base3.shape[0]}")
print(f"Tamanho do train_data após filtragem: {Base4.shape[0]}")

Tamanho original do train_data: 4963325
Tamanho do train_data após filtragem: 1153627


In [10]:
Base3.to_csv('datasets/Base3.csv', index=False)

In [34]:
# Cria a coluna de interação com base em votes_helpful e recommended
Base3['interaction'] = Base3.apply(
    lambda row: 1 + row['votes_helpful'] if row['recommended'] else -1 - row['votes_helpful'], axis=1
)

# Verifica as primeiras linhas para confirmar a criação da coluna
print(Base3[['recommended', 'votes_helpful', 'interaction']].head(10))

   recommended  votes_helpful  interaction
0        False              0           -1
1         True              0            1
2         True              0            1
3         True              0            1
4         True              0            1
5         True              0            1
6         True              0            1
7         True              0            1
8         True              1            2
9         True              0            1


In [55]:
Base4.to_csv('datasets/Base4.csv', index=False)

In [ ]:
from sklearn.preprocessing import StandardScaler
# Supondo que item_features seja criado a partir de Base4
item_features = Base4.drop(columns=['app_id', 'app_name', 'recommended', 'votes_helpful', 'author.steamid', 'author.playtime_at_review', 'interaction']).reset_index(drop=True)

# Normaliza os dados
scaler = StandardScaler()
item_features_normalized = scaler.fit_transform(item_features)

In [37]:
# Aplica PCA para reduzir a dimensionalidade
pca = PCA(n_components=20)  # Ajuste o número de componentes conforme necessário
item_features_reduced = pca.fit_transform(item_features_normalized)

In [40]:
# Multiplica cada linha pelo peso de 'interaction'
item_features_weighted = item_features_reduced * Base4['interaction'].values.reshape(-1, 1)

# Converte para uma matriz esparsa para otimizar memória
item_features_sparse = csr_matrix(item_features_weighted)


In [41]:
def calculate_similarity_for_item(item_index):
    if item_index >= item_features_sparse.shape[0]:
        print(f"Índice {item_index} fora dos limites para item_features_sparse")
        return np.array([])  # Retorna uma matriz vazia se o índice estiver fora do limite
    
    item_vector = item_features_sparse[item_index]
    similarities = cosine_similarity(item_vector, item_features_sparse)
    return similarities.flatten()


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_similarity_for_item(item_index):
    # Verifica se o item_index está dentro do limite de item_features_sparse
    if item_index >= item_features_sparse.shape[0]:
        print(f"Índice {item_index} fora dos limites para item_features_sparse")
        return np.array([])  # Retorna uma matriz vazia se o índice estiver fora do limite
    
    item_vector = item_features_sparse[item_index]
    similarities = cosine_similarity(item_vector, item_features_sparse)
    return similarities.flatten()

In [46]:
def recommend_for_user(user_id, train_data, top_n=10, min_similarity=0.2):
    user_data = train_data[(train_data['author.steamid'] == user_id) & (train_data['recommended'] == True)]
    recommended_indices = user_data.index.tolist()

    similarity_scores = {}

    for item_index in recommended_indices:
        if item_index < item_features_sparse.shape[0]:
            similarities = calculate_similarity_for_item(item_index)
            
            for idx, score in enumerate(similarities):
                if idx not in recommended_indices:
                    if idx in similarity_scores:
                        similarity_scores[idx] += score
                    else:
                        similarity_scores[idx] = score

    sorted_items = sorted(similarity_scores.items(), key=lambda x: x[1], reverse=True)
    recommended_items = []
    unique_items = set()

    for idx, _ in sorted_items:
        if idx in train_data.index:
            item_name = train_data.loc[idx, 'app_name']
            if item_name not in unique_items:
                recommended_items.append(item_name)
                unique_items.add(item_name)
        if len(recommended_items) == top_n:
            break

    return recommended_items


In [48]:
# Testa o código com um usuário específico
user_id = Base4['author.steamid'].iloc[0]  # Exemplo: pegando o primeiro usuário do conjunto filtrado
recommendations = recommend_for_user(user_id, Base4, top_n=5)
print("Recomendações com interação ponderada:", recommendations)


Recomendações com interação ponderada: ['The Elder Scrolls V: Skyrim', 'Grand Theft Auto V', 'Farm Manager 2018', 'Urban Empire', 'Pillars of Eternity II: Deadfire']


In [49]:
def recommend_for_new_user(train_data, top_n=5):
    # Calcula os itens mais populares com base no número de recomendações positivas
    popular_items = train_data[train_data['recommended'] == True]['app_name'].value_counts().head(top_n).index.tolist()
    return popular_items


In [50]:
# Exemplo de uso
print("Recomendações para usuário novo:", recommend_for_new_user(Base4))


Recomendações para usuário novo: ['Portal 2', 'Hollow Knight', 'Among Us', 'DOOM', 'Terraria']


In [51]:
def recommend_new_item_to_existing_user(new_item_features, user_id, train_data, item_features_sparse, top_n=1):
    # Seleciona os itens que o usuário já recomendou positivamente
    user_data = train_data[(train_data['author.steamid'] == user_id) & (train_data['recommended'] == True)]
    recommended_indices = user_data.index.tolist()
    
    # Calcula a média da similaridade entre o novo item e os itens que o usuário gostou
    new_item_vector = new_item_features.reshape(1, -1)
    similarities = [cosine_similarity(new_item_vector, item_features_sparse[idx]).flatten()[0] for idx in recommended_indices]
    
    # Calcula a similaridade média com os itens preferidos do usuário
    average_similarity = sum(similarities) / len(similarities) if similarities else 0
    
    # Retorna o novo item se a similaridade média for suficiente
    return new_item_features if average_similarity > 0.2 else []  # Ajuste o limite conforme necessário


In [52]:
# Suponha que o novo item tenha características similares a um item existente (ex.: o primeiro item)
new_item_features = item_features_sparse[0].toarray()  # Exemplo: usando as features do primeiro item como teste

# Testa a recomendação para um usuário antigo
user_id = Base4['author.steamid'].iloc[0]
print("Recomendação de item novo para usuário antigo:", recommend_new_item_to_existing_user(new_item_features, user_id, Base4, item_features_sparse))


Recomendação de item novo para usuário antigo: [[-0.00725503  1.26588339 -1.28072724 -0.32607347 -0.32815868 -1.03057891
   0.04688204 -0.72353807 -1.50951479  0.17445384  1.99449036  0.65643988
   0.05921747  0.17435552  0.85820809 -0.46559794  0.35181617  0.89253645
   1.29961009 -0.04566769]]
